In [9]:
# Remotion End-to-End Topic Video Runner
#
# Goal:
# - Use OpenRouter gpt-5.5 to generate content for each topic
# - Render videos using Remotion
# - Iterate topics from top to bottom
# - If one render fails, skip to next topic
# - Stop after generating 3 successful videos

In [10]:
# 1) Environment setup: Python deps + Node.js + Remotion runner project

import os
import re
import sys
import json
import shutil
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORK_ROOT = Path("/content/education_video") if IN_COLAB else Path("/Users/alyssa/Desktop/education_video")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Python deps for this notebook
pip_cmd = [sys.executable, "-m", "pip", "install", "-U", "openai", "openpyxl", "json-repair"]
if not IN_COLAB:
    pip_cmd = [sys.executable, "-m", "pip", "install", "--user", "--break-system-packages", "-U", "openai", "openpyxl", "json-repair"]
subprocess.check_call(pip_cmd)

# Ensure Node.js >= 18
node_ok = False
try:
    out = subprocess.check_output(["node", "-v"], text=True).strip()
    m = re.match(r"v(\d+)", out)
    node_ok = bool(m and int(m.group(1)) >= 18)
except Exception:
    node_ok = False

if IN_COLAB and not node_ok:
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_20.x | bash -"])
    subprocess.check_call(["apt-get", "install", "-y", "nodejs"])

if not node_ok:
    out = subprocess.check_output(["node", "-v"], text=True).strip()
    print("Node version:", out)

REMOTION_RUNNER_DIR = WORK_ROOT / "remotion_topic_runner"
REMOTION_RUNNER_DIR.mkdir(parents=True, exist_ok=True)

package_json = REMOTION_RUNNER_DIR / "package.json"
if not package_json.exists():
    package_json.write_text(json.dumps({
        "name": "remotion-topic-runner",
        "private": True,
        "version": "1.0.0",
        "scripts": {"render": "remotion render src/index.tsx TopicVideo out/video.mp4"},
        "dependencies": {
            "remotion": "latest",
            "@remotion/cli": "latest",
            "react": "latest",
            "react-dom": "latest"
        }
    }, indent=2), encoding="utf-8")

subprocess.check_call(["npm", "install"], cwd=str(REMOTION_RUNNER_DIR))
print("Setup complete:", REMOTION_RUNNER_DIR)

Setup complete: /content/education_video/remotion_topic_runner


In [11]:
# 2.5) Baseline-aligned overrides
# Multi-page settings (closer to manim baseline structure)

SLIDES_PER_VIDEO = 5
SUCCESS_TARGET = 0  # 0 = all remaining topics
print("Overrides -> SLIDES_PER_VIDEO:", SLIDES_PER_VIDEO, "SUCCESS_TARGET:", SUCCESS_TARGET)

Overrides -> SLIDES_PER_VIDEO: 5 SUCCESS_TARGET: 3


In [12]:
# 2) Runtime config + Google Drive integration

from datetime import datetime

import sys
sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
from colab_bootstrap import ensure_openai_key, setup_api_tracking, openai_model, print_api_usage

ensure_openai_key()
setup_api_tracking()
OPENAI_MODEL = openai_model()
DRIVE_INPUT_DIR = None
DRIVE_OUTPUT_ROOT = None

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/education_video")
    DRIVE_OUTPUT_ROOT = DRIVE_INPUT_DIR / "remotion_outputs"
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    TOPICS_XLSX = DRIVE_INPUT_DIR / "topics.xlsx"
else:
    TOPICS_XLSX = WORK_ROOT / "topics.xlsx"
    cloud_root = Path.home() / "Library" / "CloudStorage"
    if cloud_root.exists():
        for base in cloud_root.glob("GoogleDrive*"):
            edu = base / "My Drive" / "education_video"
            if edu.exists():
                DRIVE_INPUT_DIR = edu
                DRIVE_OUTPUT_ROOT = edu / "remotion_outputs"
                DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
                if not TOPICS_XLSX.exists() and (edu / "topics.xlsx").exists():
                    TOPICS_XLSX = edu / "topics.xlsx"
                break

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")
OUTPUT_ROOT = WORK_ROOT / "remotion_outputs" / RUN_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUCCESS_TARGET = 0  # 0 = all remaining topics
print("Model:", OPENAI_MODEL)
print("Topics:", TOPICS_XLSX)
print("Output root:", OUTPUT_ROOT)
if DRIVE_OUTPUT_ROOT is not None:
    print("Drive output root:", DRIVE_OUTPUT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model: openai/gpt-5.5
Topics: /content/drive/MyDrive/education_video/topics.xlsx
Output root: /content/education_video/remotion_outputs/run_20260707_182946
Drive output root: /content/drive/MyDrive/education_video/remotion_outputs


In [13]:
# 3) Load topics from first sheet (top to bottom)

from openpyxl import load_workbook


def _clean(v):
    return "" if v is None else str(v).strip()

wb = load_workbook(TOPICS_XLSX, read_only=True, data_only=True)
ws = wb.worksheets[0]
rows = ws.iter_rows(values_only=True)
header = [_clean(x) for x in next(rows)]

topic_rows = []
for i, row in enumerate(rows, start=2):
    item = {header[idx]: _clean(row[idx]) if idx < len(row) else "" for idx in range(len(header))}
    req_id = item.get("Request ID", "")
    topic = item.get("Topic", "")
    persona = item.get("Student Persona", "")
    if not req_id or not topic:
        continue
    if req_id == "Request ID" or topic == "Topic":
        continue
    topic_rows.append({"row": i, "request_id": req_id, "topic": topic, "persona": persona})

if not topic_rows:
    raise RuntimeError("No valid topics found.")

import sys
sys.path.insert(0, "/content/drive/MyDrive/education_video/baseline/runs")
from common_topics import topic_has_video

_output_check = DRIVE_OUTPUT_ROOT if DRIVE_OUTPUT_ROOT is not None else WORK_ROOT / "remotion_outputs"
_pending = []
for t in topic_rows:
    if topic_has_video(_output_check, t["request_id"]):
        print(f"SKIP (done): [{t['request_id']}] {t['topic']}")
        continue
    _pending.append(t)
topic_rows = _pending

if not topic_rows:
    print("All remotion topics already have videos.")
else:
    _target = len(topic_rows) if SUCCESS_TARGET == 0 else SUCCESS_TARGET
    print(f"Loaded {len(topic_rows)} pending topics. Will stop after {_target} successes.")
    for t in topic_rows[:10]:
        print(f"- [{t['request_id']}] {t['topic']}")

Loaded 68 topics. Will stop after 3 successes.
- [tm_cs_bin_001] Base-2 Number Systems and Data Representation
- [tm_cs_saf_002] Fundamentals of Personal Cybersecurity and Defense
- [tm_cs_col_003] Version Control Principles via Collaborative Workflows
- [tm_cs_ite_004] Control Flow Structures and Definite/Indefinite Loops
- [tm_cs_err_005] Systematic Debugging and Exception Classification
- [tm_cs_obj_006] Foundations of Object-Oriented Blueprint Modeling
- [tm_cs_des_007] Modular Decomposition and System Pseudocode Planning
- [tm_cs_fau_008] Redundancy and Recovery Models in Distributed Networks
- [tm_cs_dsml_009] The End-to-End Predictive Pipeline Framework
- [tm_cs_rea_010] Statistical Hypothesis Formulations and Inference


In [14]:
# 4) Create Remotion template files (stable renderer)

src_dir = REMOTION_RUNNER_DIR / "src"
src_dir.mkdir(parents=True, exist_ok=True)

(src_dir / "index.tsx").write_text(
"""import {registerRoot} from 'remotion';
import {Root} from './Root';

registerRoot(Root);
""",
encoding="utf-8",
)

(src_dir / "Root.tsx").write_text(
"""import React from 'react';
import {Composition} from 'remotion';
import {TopicVideo, TopicProps} from './TopicVideo';

export const Root: React.FC = () => {
  return (
    <>
      <Composition<TopicProps>
        id=\"TopicVideo\"
        component={TopicVideo}
        durationInFrames={900}
        fps={30}
        width={1280}
        height={720}
        defaultProps={{
          topic: 'Topic',
          subtitle: 'Subtitle',
          bullets: ['Point 1', 'Point 2', 'Point 3', 'Point 4', 'Point 5'],
        }}
      />
    </>
  );
};
""",
encoding="utf-8",
)

(src_dir / "TopicVideo.tsx").write_text(
"""import React from 'react';
import {AbsoluteFill, Easing, interpolate, spring, useCurrentFrame, useVideoConfig} from 'remotion';

export type TopicProps = {
  topic: string;
  subtitle: string;
  bullets: string[];
};

export const TopicVideo: React.FC<TopicProps> = ({topic, subtitle, bullets}) => {
  const frame = useCurrentFrame();
  const {fps} = useVideoConfig();

  const titleIn = spring({frame, fps, config: {damping: 200}});
  const subtitleIn = spring({frame: frame - 15, fps, config: {damping: 200}});

  return (
    <AbsoluteFill
      style={{
        backgroundColor: '#0f172a',
        color: 'white',
        fontFamily: 'Inter, Arial, sans-serif',
        padding: 64,
      }}
    >
      <div
        style={{
          transform: `translateY(${interpolate(titleIn, [0, 1], [30, 0])}px)`,
          opacity: titleIn,
          fontSize: 56,
          fontWeight: 800,
          lineHeight: 1.1,
          marginBottom: 20,
        }}
      >
        {topic}
      </div>

      <div
        style={{
          transform: `translateY(${interpolate(subtitleIn, [0, 1], [24, 0])}px)`,
          opacity: subtitleIn,
          fontSize: 32,
          color: '#cbd5e1',
          marginBottom: 36,
        }}
      >
        {subtitle}
      </div>

      <div style={{display: 'grid', gap: 16}}>
        {bullets.slice(0, 6).map((b, i) => {
          const local = Math.max(0, frame - (30 + i * 22));
          const p = spring({frame: local, fps, config: {damping: 150}});
          const x = interpolate(p, [0, 1], [24, 0], {easing: Easing.out(Easing.cubic)});
          return (
            <div
              key={i}
              style={{
                opacity: p,
                transform: `translateX(${x}px)`,
                fontSize: 34,
                lineHeight: 1.3,
                background: 'rgba(148, 163, 184, 0.12)',
                border: '1px solid rgba(148, 163, 184, 0.35)',
                borderRadius: 12,
                padding: '14px 18px',
              }}
            >
              {i + 1}. {b}
            </div>
          );
        })}
      </div>
    </AbsoluteFill>
  );
};
""",
encoding="utf-8",
)

print("Remotion template ready:", src_dir)

Remotion template ready: /content/education_video/remotion_topic_runner/src


In [20]:
# 5.5) Override with multi-slide Remotion template + structured generator

# Re-write template to multi-slide composition
src_dir = REMOTION_RUNNER_DIR / "src"
src_dir.mkdir(parents=True, exist_ok=True)

(src_dir / "index.tsx").write_text(
"""import {registerRoot} from 'remotion';
import {Root} from './Root';

registerRoot(Root);
""",
encoding="utf-8",
)

(src_dir / "Root.tsx").write_text(
"""import React from 'react';
import {Composition} from 'remotion';
import {TopicVideo, TopicProps} from './TopicVideo';
import {CURRENT_PROPS} from './inputProps';

export const Root: React.FC = () => {
  return (
    <>
      <Composition<TopicProps>
        id=\"TopicVideo\"
        component={TopicVideo}
        durationInFrames={1200}
        fps={30}
        width={1280}
        height={720}
        defaultProps={CURRENT_PROPS}
      />
    </>
  );
};
""",
encoding="utf-8",
)

(src_dir / "inputProps.ts").write_text(
"""import {TopicProps} from './TopicVideo';

export const CURRENT_PROPS: TopicProps = {
  topic: 'Topic',
  subtitle: 'Subtitle',
  recap: 'Recap',
  slides: [
    {title: 'Slide 1', bullets: ['A', 'B', 'C']},
    {title: 'Slide 2', bullets: ['A', 'B', 'C']},
    {title: 'Slide 3', bullets: ['A', 'B', 'C']},
    {title: 'Slide 4', bullets: ['A', 'B', 'C']},
    {title: 'Slide 5', bullets: ['A', 'B', 'C']},
  ],
};
""",
encoding="utf-8",
)

(src_dir / "TopicVideo.tsx").write_text(
"""import React from 'react';
import {AbsoluteFill, interpolate, spring, useCurrentFrame, useVideoConfig} from 'remotion';

export type TopicProps = {
  topic: string;
  subtitle: string;
  recap: string;
  slides: {title: string; bullets: string[]}[];
};

const SLIDE_FRAMES = 210;

export const TopicVideo: React.FC<TopicProps> = ({topic, subtitle, recap, slides}) => {
  const frame = useCurrentFrame();
  const {fps} = useVideoConfig();

  const introFrames = 120;
  const recapStart = introFrames + slides.length * SLIDE_FRAMES;

  const inIntro = frame < introFrames;
  const inRecap = frame >= recapStart;

  const baseStyle: React.CSSProperties = {
    backgroundColor: '#0f172a',
    color: 'white',
    fontFamily: 'Inter, Arial, sans-serif',
    padding: 64,
  };

  if (inIntro) {
    const p = spring({frame, fps, config: {damping: 200}});
    return (
      <AbsoluteFill style={baseStyle}>
        <div style={{opacity: p, transform: `translateY(${interpolate(p, [0, 1], [26, 0])}px)`, fontSize: 60, fontWeight: 800, marginBottom: 24}}>{topic}</div>
        <div style={{opacity: p, fontSize: 34, color: '#cbd5e1'}}>{subtitle}</div>
      </AbsoluteFill>
    );
  }

  if (inRecap) {
    const local = frame - recapStart;
    const p = spring({frame: local, fps, config: {damping: 170}});
    return (
      <AbsoluteFill style={baseStyle}>
        <div style={{fontSize: 52, fontWeight: 800, marginBottom: 24, opacity: p}}>Recap</div>
        <div style={{fontSize: 36, lineHeight: 1.35, color: '#e2e8f0', opacity: p}}>{recap}</div>
      </AbsoluteFill>
    );
  }

  const slideIndex = Math.min(slides.length - 1, Math.floor((frame - introFrames) / SLIDE_FRAMES));
  const local = (frame - introFrames) % SLIDE_FRAMES;
  const slide = slides[slideIndex];
  const p = spring({frame: local, fps, config: {damping: 180}});

  return (
    <AbsoluteFill style={baseStyle}>
      <div style={{fontSize: 30, color: '#94a3b8', marginBottom: 10}}>Step {slideIndex + 1} / {slides.length}</div>
      <div style={{fontSize: 52, fontWeight: 800, marginBottom: 24, opacity: p}}>{slide.title}</div>
      <div style={{display: 'grid', gap: 14}}>
        {slide.bullets.slice(0, 4).map((b, i) => {
          const bp = spring({frame: Math.max(0, local - i * 12), fps, config: {damping: 180}});
          return (
            <div key={i} style={{fontSize: 34, lineHeight: 1.3, background: 'rgba(148,163,184,0.12)', border: '1px solid rgba(148,163,184,0.35)', borderRadius: 12, padding: '12px 16px', opacity: bp, transform: `translateX(${interpolate(bp, [0, 1], [18, 0])}px)`}}>
              {i + 1}. {b}
            </div>
          );
        })}
      </div>
    </AbsoluteFill>
  );
};
""",
encoding="utf-8",
)


def build_prompt(topic: str, persona: str) -> str:
    persona_line = persona if persona else "General learners"
    return f"""
You are creating a structured educational video outline.
Return strict JSON only with keys:
- subtitle: string
- recap: string
- slides: array of exactly {SLIDES_PER_VIDEO} slide objects
Each slide object:
- title: string
- bullets: array of 3 to 4 concise bullet strings

Topic: {topic}
Audience: {persona_line}

Rules:
- progressive pedagogy from basics to application
- each slide must have different focus
- no markdown, no code fences
""".strip()


def make_props(topic: str, persona: str) -> dict:
    prompt = build_prompt(topic, persona)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1200,
        extra_headers={
            "HTTP-Referer": "https://localhost",
            "X-OpenRouter-Title": "Remotion Multi Slide Runner",
        },
    )
    content = (resp.choices[0].message.content or "").strip()
    if content.startswith("```"):
        content = re.sub(r"^```[a-zA-Z]*\n", "", content)
        content = re.sub(r"\n```$", "", content)

    data = json.loads(repair_json(content))

    subtitle = str(data.get("subtitle", "Core concepts explained step-by-step")).strip()[:120]
    recap = str(data.get("recap", "Review the key ideas and one practical takeaway.")).strip()[:240]

    raw_slides = data.get("slides", [])
    slides = []
    if isinstance(raw_slides, list):
        for s in raw_slides:
            if not isinstance(s, dict):
                continue
            title = str(s.get("title", "")).strip()[:90] or "Key idea"
            bullets = s.get("bullets", [])
            if not isinstance(bullets, list):
                bullets = []
            bullets = [str(b).strip()[:120] for b in bullets if str(b).strip()][:4]
            if len(bullets) < 3:
                bullets = [
                    "Define the concept clearly",
                    "Explain why it matters",
                    "Show a practical example",
                ]
            slides.append({"title": title, "bullets": bullets})

    while len(slides) < SLIDES_PER_VIDEO:
        i = len(slides) + 1
        slides.append({"title": f"Step {i}", "bullets": ["Core idea", "How it works", "Example"]})
    slides = slides[:SLIDES_PER_VIDEO]

    return {
        "topic": topic,
        "subtitle": subtitle,
        "recap": recap,
        "slides": slides,
    }

print("Multi-slide overrides active.")

Multi-slide overrides active.


In [19]:
# 5) OpenRouter content generation helpers

from openai import OpenAI
from json_repair import repair_json

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def safe_name(text: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_\- ]+", "", text).strip()
    s = re.sub(r"\s+", "_", s)
    return s[:80] if s else "topic"


def build_prompt(topic: str, persona: str) -> str:
    persona_line = persona if persona else "General learners"
    return f"""
You are creating educational video content.
Return strict JSON only with keys:
- subtitle: string
- bullets: array of 5 concise teaching bullets

Topic: {topic}
Audience: {persona_line}

Rules:
- bullets must be short, clear, and beginner-friendly
- no markdown, no code fences
""".strip()


def make_props(topic: str, persona: str) -> dict:
    prompt = build_prompt(topic, persona)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=600,
        extra_headers={
            "HTTP-Referer": "https://localhost",
            "X-OpenRouter-Title": "Remotion Topic Runner",
        },
    )
    content = (resp.choices[0].message.content or "").strip()
    if content.startswith("```"):
        content = re.sub(r"^```[a-zA-Z]*\n", "", content)
        content = re.sub(r"\n```$", "", content)

    data = json.loads(repair_json(content))
    subtitle = str(data.get("subtitle", "Core concepts explained simply")).strip()[:120]
    bullets = data.get("bullets", [])
    if not isinstance(bullets, list):
        bullets = []
    bullets = [str(x).strip() for x in bullets if str(x).strip()][:6]
    if len(bullets) < 3:
        bullets = [
            "What the concept means",
            "How it works step by step",
            "Common mistakes to avoid",
            "A quick practical example",
            "Key takeaway summary",
        ]

    return {"topic": topic, "subtitle": subtitle, "bullets": bullets}

print("Helpers ready.")

Helpers ready.


In [21]:
# 6) End-to-end run: top-down topics, skip failures, stop after 3 successes

results = []
success_count = 0

for idx, item in enumerate(topic_rows, start=1):
    if SUCCESS_TARGET > 0 and success_count >= SUCCESS_TARGET:
        break

    req_id = item["request_id"]
    topic = item["topic"]
    persona = item["persona"]

    topic_dir = OUTPUT_ROOT / f"{idx:03d}_{req_id}_{safe_name(topic)}"
    topic_dir.mkdir(parents=True, exist_ok=True)

    props_path = topic_dir / "props.json"
    out_video = topic_dir / "video.mp4"
    log_path = topic_dir / "render.log"

    print("\n" + "=" * 80)
    print(f"Trying topic #{idx}: {topic}")

    try:
        props = make_props(topic, persona)
        props_path.write_text(json.dumps(props, ensure_ascii=False, indent=2), encoding="utf-8")

        input_props_ts = REMOTION_RUNNER_DIR / "src" / "inputProps.ts"
        input_props_ts.write_text(
            "import {TopicProps} from './TopicVideo';\n\n"
            + "export const CURRENT_PROPS: TopicProps = "
            + json.dumps(props, ensure_ascii=False, indent=2)
            + ";\n",
            encoding="utf-8",
        )

        print("Rendering with props title:", props.get("topic"), "slides:", len(props.get("slides", [])))

        cmd = [
            "npx",
            "remotion",
            "render",
            "src/index.tsx",
            "TopicVideo",
            str(out_video),
            "--overwrite",
        ]
        proc = subprocess.run(cmd, cwd=str(REMOTION_RUNNER_DIR), capture_output=True, text=True)
        log_path.write_text(
            "CMD: " + " ".join(cmd) + "\n\nSTDOUT:\n" + (proc.stdout or "") + "\n\nSTDERR:\n" + (proc.stderr or ""),
            encoding="utf-8",
        )

        if proc.returncode != 0 or not out_video.exists():
            raise RuntimeError(f"Render failed (exit={proc.returncode})")

        success_count += 1
        results.append(
            {
                "request_id": req_id,
                "topic": topic,
                "status": "ok",
                "video": str(out_video),
                "folder": str(topic_dir),
            }
        )
        print(f"SUCCESS {success_count}/{SUCCESS_TARGET}: {out_video}")

    except Exception as e:
        results.append(
            {
                "request_id": req_id,
                "topic": topic,
                "status": "failed",
                "error": f"{type(e).__name__}: {e}",
                "folder": str(topic_dir),
            }
        )
        print("FAILED, skip to next:", e)

summary_path = OUTPUT_ROOT / "summary.json"
summary_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("\nSummary:", summary_path)
print(f"Successful videos: {success_count}")

if DRIVE_OUTPUT_ROOT is not None:
    drive_run_dir = DRIVE_OUTPUT_ROOT / OUTPUT_ROOT.name
    if drive_run_dir.exists():
        shutil.rmtree(drive_run_dir)
    shutil.copytree(OUTPUT_ROOT, drive_run_dir)

    zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT), "zip", root_dir=OUTPUT_ROOT))
    shutil.copy2(zip_path, DRIVE_OUTPUT_ROOT / zip_path.name)
    print("Synced to Drive:", drive_run_dir)
    print("Synced zip:", DRIVE_OUTPUT_ROOT / zip_path.name)


Trying topic #1: Base-2 Number Systems and Data Representation
Rendering with props title: Base-2 Number Systems and Data Representation slides: 5
SUCCESS 1/3: /content/education_video/remotion_outputs/run_20260707_182946/001_tm_cs_bin_001_Base-2_Number_Systems_and_Data_Representation/video.mp4

Trying topic #2: Fundamentals of Personal Cybersecurity and Defense
Rendering with props title: Fundamentals of Personal Cybersecurity and Defense slides: 5
SUCCESS 2/3: /content/education_video/remotion_outputs/run_20260707_182946/002_tm_cs_saf_002_Fundamentals_of_Personal_Cybersecurity_and_Defense/video.mp4

Trying topic #3: Version Control Principles via Collaborative Workflows
Rendering with props title: Version Control Principles via Collaborative Workflows slides: 5
SUCCESS 3/3: /content/education_video/remotion_outputs/run_20260707_182946/003_tm_cs_col_003_Version_Control_Principles_via_Collaborative_Workflows/video.mp4

Summary: /content/education_video/remotion_outputs/run_20260707_18

In [22]:
# 7) Quick view

ok = [r for r in results if r.get("status") == "ok"]
failed = [r for r in results if r.get("status") != "ok"]

print(f"Output root: {OUTPUT_ROOT}")
print(f"Success: {len(ok)}")

if ok:
    print("\nVideos:")
    for r in ok:
        print("-", r["video"])

if failed:
    print("\nFailed topics (skipped):")
    for r in failed[:10]:
        print("-", r["topic"], "=>", r.get("error", "unknown"))

Output root: /content/education_video/remotion_outputs/run_20260707_182946
Success: 3

Videos:
- /content/education_video/remotion_outputs/run_20260707_182946/001_tm_cs_bin_001_Base-2_Number_Systems_and_Data_Representation/video.mp4
- /content/education_video/remotion_outputs/run_20260707_182946/002_tm_cs_saf_002_Fundamentals_of_Personal_Cybersecurity_and_Defense/video.mp4
- /content/education_video/remotion_outputs/run_20260707_182946/003_tm_cs_col_003_Version_Control_Principles_via_Collaborative_Workflows/video.mp4
